## Aim

In this code, the first part of the GEE Javascript code [here](https://github.com/EllenB/lstroad/blob/main/scripts/04_indices_chennai.js) is translated into Python. This code computes several indices at the ward level;

- NDVI: Normalised Difference Vegetation Index
- NDBI:  Normalised Difference Building Index
- MNDWI: Modified Normalised Difference Water Index
- BSI: Bareness Soil Index

## Importing the libraries

In [1]:
from pathlib import Path

import ee
import geemap
import geopandas as gpd
import pandas as pd

In [ ]:
cloud_project = 'FILL IN YOUR CLOUD PROJECT'

try:
    ee.Initialize(project=cloud_project)
except:
    ee.Authenticate()
    ee.Initialize(project=cloud_project)

## Reading in and preparing the data

In [3]:
# Define the paths
BASE_DIR = Path.cwd().parent
print(BASE_DIR)

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

C:\Ellen\personal\projects\geospatial\lst_bang_chennai\lstroad


In [4]:
## Detailed wards
df_inpath = PROCESSED_DIR / "chennai_fix_mapshaper.shp"
print(df_inpath)

C:\Ellen\personal\projects\geospatial\lst_bang_chennai\lstroad\data\processed\chennai_fix_mapshaper.shp


In [5]:
# https://geemap.org/notebooks/10_shapefiles/ 
# Convert to something geemap can understand
chennai = geemap.shp_to_ee(df_inpath)

In [6]:
print(f'The number of wards in Chennai is {chennai.size().getInfo()}')

The number of wards in Chennai is 200


In [7]:
chennaiGeometry = chennai.union(50)

Map = geemap.Map()
Map.centerObject(chennaiGeometry, zoom=10)
Map # Uncomment if you want to see the map. 
## Otherwise you can wait till later Maps as all layers are in them

Map(center=[13.063596024256674, 80.231179751739], controls=(WidgetControl(options=['position', 'transparent_bg…

In [8]:
Map.addLayer(chennaiGeometry, {}, 'Chennai outline')
Map

Map(bottom=121776.0, center=[13.063596024256674, 80.231179751739], controls=(WidgetControl(options=['position'…

In [9]:
## Add the ward data
image = ee.Image().paint(chennai, 0, 2)
Map.addLayer(image, {'palette': 'red'}, "Bangalore geometry")
Map

Map(bottom=121776.0, center=[13.063596024256674, 80.231179751739], controls=(WidgetControl(options=['position'…

## Importing the Landsat collection

In [10]:
colThsummer = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
.filterBounds(chennaiGeometry) \
.filterDate('2024-01-01', '2025-11-01') \
.filter(ee.Filter.calendarRange(3, 5, 'month'))

## colThsummer

In [11]:
def cloud_rescale(image):

    # This is the cloud mask we have been "using":
    # Code obtained "from":
    # "https":#courses.spatialthoughts.com/end-to-end-gee-supplement.html#derive-lst-from-landsat-images
    qaMask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturationMask = image.select('QA_RADSAT').eq(0)

    # scaling
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    return image.addBands(opticalBands, None, True) \
    .addBands(thermalBands, None, True) \
    .updateMask(qaMask) \
    .updateMask(saturationMask)

colThsummer = colThsummer.map(cloud_rescale)

## Computing the indices

In [12]:
def addIndices(image):

    ndvi = image.normalizedDifference(['SR_B5','SR_B4']).rename('NDVI')
    ndbi = image.normalizedDifference(['SR_B6','SR_B5']).rename('NDBI')
    mndwi = image.normalizedDifference(['SR_B3','SR_B6']).rename('MNDWI')
    bsi = image.expression(
    '((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE))',
    {
        "SWIR": image.select('SR_B6'),
        "RED":  image.select('SR_B4'),
        "NIR":  image.select('SR_B5'),
        "BLUE": image.select('SR_B2')
    }
    ).rename('BSI')

    return image.addBands([ndvi, ndbi, mndwi, bsi])

In [13]:
colThsummer = colThsummer.map(addIndices)

In [14]:
## colThsummer

In [15]:
# Median indices
# Take median of the indices across all images
medianIndices = colThsummer \
.select(['NDVI','NDBI','MNDWI', 'BSI']) \
.median() \
.clip(chennaiGeometry)

In [16]:
## medianIndices

## Visualisation of the indices

In [17]:
# NDVI
ndviVis = {
    "min": 0,
    "max": 1,
    "palette": ['red','yellow','green']
}

# Add to map
Map.addLayer(medianIndices.select('NDVI'), ndviVis, 'Median NDVI')
Map

Map(bottom=121776.0, center=[13.063596024256674, 80.231179751739], controls=(WidgetControl(options=['position'…

In [18]:
# NDBI visualization
ndbiVis = {"min":-0.5, "max":0.5, "palette":['white','red']}
Map.addLayer(medianIndices.select('NDBI'), ndbiVis, 'Median NDBI')
Map

Map(bottom=121776.0, center=[13.063596024256674, 80.231179751739], controls=(WidgetControl(options=['position'…

In [19]:
mndwiVis = {"min":0, "max":0.5, "palette":['white','blue']}
Map.addLayer(medianIndices.select('MNDWI'), mndwiVis, 'Median MNDWI')
Map

Map(bottom=121776.0, center=[13.063596024256674, 80.231179751739], controls=(WidgetControl(options=['position'…

In [20]:
# BSI visualization
bsiVis = {"min":-0.2, "max":0.3, "palette":['green','yellow','red']}
Map.addLayer(medianIndices.select('BSI'), bsiVis, 'Median BSI')
Map

Map(bottom=121776.0, center=[13.063596024256674, 80.231179751739], controls=(WidgetControl(options=['position'…

## Zonal statistics

In [21]:
indiceswards = medianIndices.reduceRegions(
collection=chennai,
reducer=ee.Reducer.mean(),
scale=30,
)

In [22]:
## indiceswards

In [23]:
outdir = PROCESSED_DIR /'indices_wards_geemap_chennai.csv'
print(outdir)

C:\Ellen\personal\projects\geospatial\lst_bang_chennai\lstroad\data\processed\indices_wards_geemap_chennai.csv


In [24]:
indiceswards = indiceswards.select(['BSI', 'MNDWI', 'NDBI', 'NDVI', 'Name', 'zone_id', 'zone_name'])

In [25]:
geemap.ee_export_vector(indiceswards, outdir, verbose=True)

Generating URL ...
Please wait ...
Data downloaded to C:\Ellen\personal\projects\geospatial\lst_bang_chennai\lstroad\data\processed\indices_wards_geemap_chennai.csv


## Visualisation

## References

Becker, S. J., Maloney, M. C., Griffin, A. W., Lasko, K., & Sussman, H. S. (2025). Bare ground classification using a spectral index ensemble and machine learning models optimized across 12 international study sites. Geocarto International, 40(1), 2465452.

Roy, B., & Bari, E. (2022). Examining the relationship between land surface temperature and landscape features using spectral indices with Google Earth Engine. Heliyon, 8(9).